# Mantenimiento predictivo — NASA C-MAPSS FD001
## Análisis exploratorio, inferencia estadística y estimación de RUL

**Objetivo:** estimar la vida útil remanente (Remaining Useful Life, RUL) de motores turbofán a partir de variables operativas y sensores.

> Ejecuta el notebook desde la carpeta raíz del repositorio. Los archivos originales deben estar en `data/raw/`. No se reportan resultados numéricos hasta ejecutar el análisis con los datos reales.


In [ ]:
# Importaciones y configuración reproducible
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, mannwhitneyu
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from pipeline import load_data, add_targets, etl, nasa_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


In [ ]:
# 1. Carga de datos: se usan train, test y RUL oficial.
train_raw, test_raw, rul_test = load_data()
print("Train:", train_raw.shape, "| Test:", test_raw.shape, "| RUL test:", rul_test.shape)
display(train_raw.head())
display(test_raw.head())


In [ ]:
# 2. Auditoría ETL: nulos, duplicados, sensores constantes y valores extremos.
train_clean, test_clean, features, audit = etl(train_raw, test_raw)
print(json.dumps(audit, indent=2, ensure_ascii=False))
print("Sensores predictivos retenidos:", features)

# No se imputan ni eliminan valores extremos automáticamente: pueden representar degradación.
# La imputación por mediana se incorpora dentro de cada pipeline para evitar fuga de información.


In [ ]:
# 3. Construcción de la variable objetivo RUL.
train = add_targets(train_clean, cap=125)
display(train[["unit_number", "time_in_cycles", "RUL_unbounded", "RUL"]].head())
print("RUL sin límite:", train.RUL_unbounded.describe())
print("RUL limitada:", train.RUL.describe())


## 4. Análisis exploratorio (EDA)

Los sensores y ajustes operativos son variables del simulador C-MAPSS. No se asigna una unidad física (°C, psi, rpm, etc.) sin una tabla de metadatos que lo respalde. Por ello, los ejes indican **unidad del simulador**.


In [ ]:
# Distribución de ciclos de vida por motor.
life = train.groupby("unit_number")["time_in_cycles"].max().rename("ciclos_totales")
fig, ax = plt.subplots(figsize=(9,4))
sns.histplot(life, bins=25, kde=True, ax=ax)
ax.set(title="Distribución de ciclos de vida por motor", xlabel="Ciclos hasta fallo (ciclos)", ylabel="Número de motores")
plt.show()

# Distribuciones individuales de sensores retenidos.
train[features].hist(figsize=(16,12), bins=35, layout=(-1,4), edgecolor="white")
plt.suptitle("Distribuciones de sensores — FD001", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Matriz de correlación de Spearman: útil para asociaciones monótonas no necesariamente lineales.
corr = train[features + ["RUL"]].corr(method="spearman")
plt.figure(figsize=(14,10))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=.2)
plt.title("Correlación de Spearman entre sensores y RUL")
plt.tight_layout()
plt.show()
display(corr["RUL"].drop("RUL").sort_values())


In [ ]:
# Tendencias por RUL: agrupar en bandas para reducir ruido ciclo a ciclo.
train["RUL_band"] = pd.cut(train["RUL"], bins=[-1,25,50,75,100,125], labels=["0–25","26–50","51–75","76–100","101–125"])
selected_sensors = corr["RUL"].drop("RUL").abs().sort_values(ascending=False).head(min(6,len(features))).index.tolist()
fig, axes = plt.subplots(len(selected_sensors), 1, figsize=(10, 3*len(selected_sensors)), squeeze=False)
for ax, s in zip(axes[:,0], selected_sensors):
    sns.lineplot(data=train, x="RUL_band", y=s, errorbar=("ci", 95), ax=ax)
    ax.set(title=f"{s} frente a bandas de RUL", xlabel="RUL (ciclos)", ylabel="Lectura (unidad del simulador)")
plt.tight_layout()
plt.show()


In [ ]:
# Prueba no paramétrica: compara valores del sensor en el inicio y final de vida.
# Se usa Mann–Whitney U porque no exige normalidad; la comparación es exploratoria.
# Inicio: RUL 101–125; final: RUL 0–25. Se informa p-value y tamaño de muestra.
test_rows = []
for s in features:
    early = train.loc[train.RUL.between(101,125), s].dropna()
    late = train.loc[train.RUL.between(0,25), s].dropna()
    if len(early) >= 5 and len(late) >= 5:
        stat, pval = mannwhitneyu(early, late, alternative="two-sided")
        rho, rho_p = spearmanr(train[s], train["RUL"], nan_policy="omit")
        test_rows.append({"sensor":s, "n_inicio":len(early), "n_final":len(late),
                          "U":stat, "p_value":pval, "spearman_rho":rho, "spearman_p":rho_p})
stats = pd.DataFrame(test_rows).sort_values("p_value")
display(stats)


**Interpretación estadística:** un p-value pequeño sugiere que las distribuciones entre la banda inicial y final difieren; no demuestra causalidad ni identifica por sí solo un componente físico. Se recomienda corregir comparaciones múltiples (por ejemplo, Benjamini–Hochberg) antes de afirmar significancia confirmatoria.


## 5. Modelado y validación sin data leakage

La partición se realiza por `unit_number`: todas las observaciones de un motor pertenecen a un único subconjunto. El conjunto oficial `test_FD001` se evalúa usando la última observación de cada motor y la RUL correspondiente en `RUL_FD001.txt`.


In [ ]:
# Partición por motores completos: ninguna unidad aparece en train y validación simultáneamente.
X = train[features]
y = train["RUL"]
groups = train["unit_number"]
splitter = GroupShuffleSplit(n_splits=1, test_size=.20, random_state=RANDOM_STATE)
itr, iva = next(splitter.split(X, y, groups))
Xtr, Xva, ytr, yva = X.iloc[itr], X.iloc[iva], y.iloc[itr], y.iloc[iva]
gtr, gva = groups.iloc[itr], groups.iloc[iva]
assert set(gtr).isdisjoint(set(gva)), "Error: hay motores compartidos entre train y validación."
print("Motores entrenamiento:", gtr.nunique(), "| validación:", gva.nunique())


In [ ]:
# Tres familias de modelos: baseline lineal, Random Forest y Gradient Boosting.
models = {
 "Regresión lineal": Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler()),("model",LinearRegression())]),
 "Random Forest": Pipeline([("imputer",SimpleImputer(strategy="median")),("model",RandomForestRegressor(n_estimators=250,random_state=42,n_jobs=-1))]),
 "Gradient Boosting": Pipeline([("imputer",SimpleImputer(strategy="median")),("model",HistGradientBoostingRegressor(random_state=42))])
}
results, fitted = {}, {}
for name, model in models.items():
    model.fit(Xtr,ytr)
    pred = np.clip(model.predict(Xva),0,125)
    fitted[name] = model
    results[name] = {"MAE":mean_absolute_error(yva,pred),
                     "RMSE":mean_squared_error(yva,pred)**.5,
                     "R2":r2_score(yva,pred),
                     "NASA_score":nasa_score(yva,pred)}
results_df = pd.DataFrame(results).T.sort_values("RMSE")
display(results_df)


In [ ]:
# Validación cruzada agrupada en el subconjunto de entrenamiento.
# GroupKFold impide que el mismo motor se distribuya entre folds.
gkf = GroupKFold(n_splits=5)
rf_cv = Pipeline([("imputer",SimpleImputer(strategy="median")),
                  ("model",RandomForestRegressor(n_estimators=200,random_state=42,n_jobs=-1))])
cv_scores = cross_val_score(rf_cv, Xtr, ytr, groups=gtr, cv=gkf,
                            scoring="neg_root_mean_squared_error", n_jobs=-1)
print("RMSE por fold:", -cv_scores)
print("RMSE CV medio:", (-cv_scores).mean(), "±", cv_scores.std())


In [ ]:
# Ajuste de hiperparámetros mediante GridSearchCV con GroupKFold.
from sklearn.model_selection import GridSearchCV
grid_model = Pipeline([("imputer",SimpleImputer(strategy="median")),
                       ("model",RandomForestRegressor(random_state=42,n_jobs=-1))])
param_grid = {"model__n_estimators":[150,300],
              "model__max_depth":[None,12],
              "model__min_samples_leaf":[1,3]}
search = GridSearchCV(grid_model,param_grid,cv=gkf,
                      scoring="neg_root_mean_squared_error",n_jobs=-1)
search.fit(Xtr,ytr,groups=gtr)
print("Mejores parámetros:",search.best_params_)
print("Mejor RMSE CV:",-search.best_score_)


In [ ]:
# Selección del modelo por RMSE de validación; predicción limitada al rango físico objetivo [0,125].
best_name = results_df.index[0]
best_model = fitted[best_name]
val_pred = np.clip(best_model.predict(Xva),0,125)


In [ ]:
# Análisis de residuos: errores frente a predicción y distribución de errores.
residuals = yva.to_numpy() - val_pred
fig, axes = plt.subplots(1,2,figsize=(13,4))
axes[0].scatter(val_pred,residuals,alpha=.35)
axes[0].axhline(0,linestyle="--",color="black")
axes[0].set(xlabel="RUL predicha (ciclos)",ylabel="Residuo real − predicho",title="Residuos frente a predicción")
sns.histplot(residuals,kde=True,ax=axes[1])
axes[1].set(title="Distribución de residuos",xlabel="Residuo (ciclos)")
plt.tight_layout()
plt.show()
print("Media residuos:", residuals.mean(), "| desviación:", residuals.std())


In [ ]:
# Importancia por permutación en validación: mide cuánto empeora RMSE al alterar cada sensor.
perm = permutation_importance(best_model,Xva,yva,n_repeats=8,random_state=42,
                              scoring="neg_root_mean_squared_error",n_jobs=-1)
importance = pd.Series(perm.importances_mean,index=features).sort_values()
importance.tail(15).plot(kind="barh",figsize=(8,6),title="Importancia por permutación")
plt.xlabel("Incremento de error al permutar")
plt.tight_layout()
plt.show()


## 6. Evaluación oficial sobre test_FD001 y RUL_FD001

La evaluación oficial usa una sola fila por motor: su último ciclo observado. La fila i de `RUL_FD001.txt` corresponde al motor i en el orden de unidades de test, por lo que se ordena explícitamente por `unit_number`.


In [ ]:
# Preparar la última observación de cada motor del conjunto test.
last_test = test_clean.sort_values(["unit_number","time_in_cycles"]).groupby("unit_number").tail(1).sort_values("unit_number")
assert len(last_test) == len(rul_test), "No coincide el número de motores test con RUL."
Xtest = last_test[features]
ytest = rul_test["RUL"].to_numpy()
test_pred = np.clip(best_model.predict(Xtest),0,125)
official = {"MAE":mean_absolute_error(ytest,test_pred),
            "RMSE":mean_squared_error(ytest,test_pred)**.5,
            "R2":r2_score(ytest,test_pred),
            "NASA_score":nasa_score(ytest,test_pred)}
print("Modelo de validación seleccionado:",best_name)
print("Métricas oficiales FD001:",official)
predictions = pd.DataFrame({"unit_number":last_test.unit_number.to_numpy(),
                            "RUL_real":ytest,"RUL_predicha":test_pred,
                            "error":test_pred-ytest})
display(predictions.head(15))


In [ ]:
# Gráfico por motor, no por índice arbitrario de fila.
plot_df = predictions.sort_values("unit_number")
fig = plt.figure(figsize=(12,5))
plt.plot(plot_df.unit_number,plot_df.RUL_real,marker="o",label="RUL real")
plt.plot(plot_df.unit_number,plot_df.RUL_predicha,marker="x",label="RUL predicha")
plt.xlabel("Número de motor (unit_number)")
plt.ylabel("RUL (ciclos)")
plt.title("Evaluación oficial: RUL real vs predicha por motor")
plt.legend()
plt.tight_layout()
plt.show()


## 7. Interpretación aeronáutica y recomendaciones

- C-MAPSS es un banco de pruebas de simulación de motores turbofán. Los sensores pueden reflejar condiciones termodinámicas y de operación, pero el número de sensor no basta para afirmar que corresponde al compresor de alta presión, turbina u otro módulo.
- La correlación, tendencia e importancia del modelo sirven para priorizar variables de seguimiento; no sustituyen límites de mantenimiento aprobados ni diagnósticos de ingeniería.
- La RUL estimada es una herramienta de apoyo. Una predicción baja debe activar revisión técnica según el programa de mantenimiento aprobado, análisis de tendencias y documentación del fabricante.
- No se deben fijar intervalos de boroscopia ni liberar/retirar motores únicamente con este modelo. Para convertir RUL en una recomendación de inspección se requiere validación con datos de flota real, incertidumbre, límites de fabricante y aprobación de ingeniería/autoridad.


## 8. Exportación y reproducibilidad

El pipeline de `src/pipeline.py` ejecuta el flujo completo y guarda auditoría, métricas, predicciones y modelo. El dashboard Streamlit permite filtrar por motor y sensor.


In [ ]:
# Guardar resultados exploratorios y modelo de notebook.
from pathlib import Path
import joblib
(ROOT/"reports").mkdir(exist_ok=True)
(ROOT/"models").mkdir(exist_ok=True)
results_df.to_csv(ROOT/"reports"/"validation_metrics_notebook.csv")
predictions.to_csv(ROOT/"data"/"processed"/"notebook_test_predictions.csv",index=False)
joblib.dump({"model":best_model,"features":features,"name":best_name},ROOT/"models"/"notebook_model.joblib")
print("Archivos guardados en reports/, models/ y data/processed/.")
